# Seq2Seq English to Hindi Translation (FAST VECTORIZED VERSION)
This notebook utilizes Vectorized Teacher Forcing to offload sequence processing directly to cuDNN and Tensor Cores, massively accelerating training speed.


## 1. Import Libraries & GPU Setup
Load TensorFlow and configure GPU memory growth.


In [1]:
import tensorflow as tf
import numpy as np
import pandas as pd
import json
import collections
import tensorflow.keras.preprocessing.sequence as ps

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("GPU Memory Growth Enabled.")
    except RuntimeError as e:
        print(e)

tf.random.set_seed(42)
np.random.seed(42)


I0000 00:00:1783413011.010614    5733 cuda_dnn.cc:8703] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
I0000 00:00:1783413011.014899    5733 cuda_blas.cc:1413] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1783413011.025749    5733 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783413011.025765    5733 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783413011.025766    5733 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783413011.025767    5733 computation_placer.cc:177] computation placer already registered. Please check linka

GPU Memory Growth Enabled.


## 2. Load Dataset
We read the dataset using Pandas.


In [2]:
data = pd.read_csv('Dataset_English_Hindi.csv')
data.columns = ['English', 'Hindi']
data = data.dropna()

english_sentences = data['English'].astype(str).tolist()
hindi_sentences = data['Hindi'].astype(str).tolist()

print(f"Total sentences: {len(english_sentences)}")


Total sentences: 130162


## 3. Build Vocabularies
Vocabulary is capped at 10,000 to maintain VRAM stability.


In [3]:
def build_vocab(sentences, max_vocab_size=10000):
    vocab = {'<PAD>': 0, '<SOS>': 1, '<EOS>': 2, '<UNK>': 3}
    next_index = 4
    
    word_freq = collections.Counter()
    for sentence in sentences:
        word_freq.update(sentence.split())
        
    common_words = word_freq.most_common(max_vocab_size - len(vocab))
    
    for word, _ in common_words:
        vocab[word] = next_index
        next_index += 1
        
    return vocab

print("Building vocabularies...")
eng_vocab = build_vocab(english_sentences, max_vocab_size=10000)
hind_vocab = build_vocab(hindi_sentences, max_vocab_size=10000)

with open('eng_vocab.json', 'w', encoding='utf-8') as f:
    json.dump(eng_vocab, f, ensure_ascii=False)
with open('hind_vocab.json', 'w', encoding='utf-8') as f:
    json.dump(hind_vocab, f, ensure_ascii=False)



Building vocabularies...


## 4. Sequence Padding
Convert sentences to integer sequences. Unknown words use `<UNK>`. We cap length to 30.


In [4]:
def sentence_to_number(sentence, vocab):
    words = sentence.split()
    unk_id = vocab['<UNK>']
    numbers = [vocab.get(word, unk_id) for word in words]
    return [vocab['<SOS>']] + numbers + [vocab['<EOS>']]

def prepare_data(eng_sentences, hind_sentences, eng_vocab, hind_vocab, max_length=30):
    src_number = [sentence_to_number(sent, eng_vocab) for sent in eng_sentences]
    tgt_number = [sentence_to_number(sent, hind_vocab) for sent in hind_sentences]

    src_padded = ps.pad_sequences(src_number, padding='post', maxlen=max_length, truncating='post', value=eng_vocab['<PAD>'])
    tgt_padded = ps.pad_sequences(tgt_number, padding='post', maxlen=max_length, truncating='post', value=hind_vocab['<PAD>'])
    return src_padded, tgt_padded

src_data, tgt_data = prepare_data(english_sentences, hindi_sentences, eng_vocab, hind_vocab)



## 5. Batching with tf.data.Dataset


In [5]:
BATCH_SIZE = 32 
BUFFER_SIZE = 10000 

dataset = tf.data.Dataset.from_tensor_slices((src_data, tgt_data))
dataset = dataset.shuffle(BUFFER_SIZE, seed=42).batch(BATCH_SIZE, drop_remainder=True)



I0000 00:00:1783413017.548215    5733 gpu_device.cc:2018] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5233 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 5060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 12.0


## 6. Model Architecture
Below we define the Encoder, Decoder, and the Vectorized Seq2Seq wrapper.


### 6.1 The Encoder


In [6]:
class Encoder(tf.keras.Model):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super(Encoder, self).__init__()
        self.embedding = tf.keras.layers.Embedding(vocab_size, embed_size)
        self.lstm = tf.keras.layers.LSTM(hidden_size, return_state=True)

    def call(self, x):
        embedded = self.embedding(x)
        _, hidden, cell = self.lstm(embedded)
        return hidden, cell



### 6.2 The Decoder


In [7]:
class Decoder(tf.keras.Model):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super(Decoder, self).__init__()
        self.embedding = tf.keras.layers.Embedding(vocab_size, embed_size)
        self.lstm = tf.keras.layers.LSTM(hidden_size, return_state=True, return_sequences=True)
        self.dense = tf.keras.layers.Dense(vocab_size)

    def call(self, x, hidden, cell):
        embedded = self.embedding(x)
        lstm_output, hidden, cell = self.lstm(embedded, initial_state=[hidden, cell])
        output = self.dense(lstm_output)
        return output, hidden, cell



### 6.3 The Vectorized Seq2Seq Wrapper
This version completely bypasses loops during training, feeding the entire sequence into cuDNN at once.


In [8]:
class Seq2Seq(tf.keras.Model):
    def __init__(self, encoder, decoder, tgt_vocab_size):
        super(Seq2Seq, self).__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.tgt_vocab_size = tgt_vocab_size

    def call(self, inputs, training=False):
        src, tgt = inputs[0], inputs[1]
        hidden, cell = self.encoder(src)
        
        if training:
            # FAST PATH: Vectorized Teacher Forcing
            # We feed the entire sequence (except the last token) in one single pass!
            # cuDNN processes the entire sequence in C++ using Tensor Cores instantly.
            decoder_input = tgt[:, :-1]
            output, _, _ = self.decoder(decoder_input, hidden, cell)
            return output
            
        else:
            # SLOW PATH: Autoregressive Loop for Inference (Real-world translation)
            # We must use a loop here because we don't know the future tokens in advance!
            batch_size = tf.shape(src)[0]
            tgt_len = tf.shape(tgt)[1] 
            
            i0 = tf.constant(1)
            input_token0 = tgt[:, 0:1] 
            all_outputs0 = tf.TensorArray(tf.float32, size=tgt_len - 1)
    
            def cond(i, input_token, h, c, all_outputs):
                return i < tgt_len
                
            def body(i, input_token, h, c, all_outputs):
                output, h, c = self.decoder(input_token, h, c)
                all_outputs = all_outputs.write(i - 1, output)
                next_input = tf.argmax(output, axis=-1, output_type=tf.int32)
                return i + 1, next_input, h, c, all_outputs
    
            _, _, _, _, final_outputs = tf.while_loop(
                cond, body,
                loop_vars=[i0, input_token0, hidden, cell, all_outputs0]
            )
    
            outputs_stacked = final_outputs.stack()
            outputs_stacked = tf.squeeze(outputs_stacked, axis=2) 
            return tf.transpose(outputs_stacked, perm=[1, 0, 2]) 



## 7. Initialize Model


In [9]:
embed_size = 256
hidden_size = 512
input_vocab_size = len(eng_vocab)
output_vocab_size = len(hind_vocab)

encoder = Encoder(input_vocab_size, embed_size, hidden_size)
decoder = Decoder(output_vocab_size, embed_size, hidden_size)
model = Seq2Seq(encoder, decoder, tgt_vocab_size=output_vocab_size)

# Compile dummy data to build both Graph paths
dummy_src = tf.zeros((BATCH_SIZE, 30), dtype=tf.int32)
dummy_tgt = tf.zeros((BATCH_SIZE, 30), dtype=tf.int32)
_ = model([dummy_src, dummy_tgt], training=True)
_ = model([dummy_src, dummy_tgt], training=False)



I0000 00:00:1783413019.329853    6074 cuda_dnn.cc:529] Loaded cuDNN version 92302


## 8. Training Logic


In [10]:
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True, reduction='none')

@tf.function
def train_step(src, tgt):
    with tf.GradientTape() as tape:
        predictions = model([src, tgt], training=True)
        target_labels = tgt[:, 1:] 

        mask = tf.cast(target_labels != hind_vocab["<PAD>"], tf.float32)
        loss = loss_fn(target_labels, predictions)
        loss = loss * mask

        total_loss = tf.reduce_sum(loss)
        total_token = tf.reduce_sum(mask)
        mean_loss = total_loss / total_token

    gradients = tape.gradient(mean_loss, model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))
    return mean_loss



## 9. Execute Training Loop with Early Stopping


In [11]:
EPOCHS = 30
PATIENCE = 3
best_loss = float('inf')
patience_counter = 0

print('Starting Fast Vectorized Training...')

for epoch in range(EPOCHS):
    total_loss = 0.0
    steps = 0
    for batch_src, batch_tgt in dataset:
        loss = train_step(batch_src, batch_tgt)
        total_loss += float(loss.numpy())
        steps += 1
        
        if steps % 500 == 0:
            print(f"Epoch {epoch+1}, Step {steps}, Loss: {loss.numpy():.4f}")
            
    avg_loss = total_loss / steps
    print(f'\nEpoch: {epoch+1} | Average Loss: {avg_loss:.4f}')
    
    if avg_loss < best_loss:
        best_loss = avg_loss
        patience_counter = 0
        model.save_weights('seq2seq_weights_cuDNN.weights.h5')
        print(f"[*] Loss improved to {best_loss:.4f}. Model weights saved.\n")
    else:
        patience_counter += 1
        print(f"[!] No improvement in loss. Patience: {patience_counter}/{PATIENCE}\n")
        if patience_counter >= PATIENCE:
            print("=> Early stopping triggered! Training stopped.")
            break



Starting Fast Vectorized Training...
Epoch 1, Step 500, Loss: 5.6076
Epoch 1, Step 1000, Loss: 5.5165
Epoch 1, Step 1500, Loss: 4.8572
Epoch 1, Step 2000, Loss: 5.0167
Epoch 1, Step 2500, Loss: 4.7540
Epoch 1, Step 3000, Loss: 4.7136
Epoch 1, Step 3500, Loss: 4.6553
Epoch 1, Step 4000, Loss: 4.4650

Epoch: 1 | Average Loss: 5.1298
[*] Loss improved to 5.1298. Model weights saved.

Epoch 2, Step 500, Loss: 4.1259
Epoch 2, Step 1000, Loss: 4.4547
Epoch 2, Step 1500, Loss: 4.5382
Epoch 2, Step 2000, Loss: 4.3017
Epoch 2, Step 2500, Loss: 4.1609
Epoch 2, Step 3000, Loss: 3.9069
Epoch 2, Step 3500, Loss: 3.8854
Epoch 2, Step 4000, Loss: 3.8518

Epoch: 2 | Average Loss: 4.2039
[*] Loss improved to 4.2039. Model weights saved.

Epoch 3, Step 500, Loss: 3.9148
Epoch 3, Step 1000, Loss: 3.8465
Epoch 3, Step 1500, Loss: 3.8395
Epoch 3, Step 2000, Loss: 3.6429
Epoch 3, Step 2500, Loss: 3.2644
Epoch 3, Step 3000, Loss: 3.8639
Epoch 3, Step 3500, Loss: 3.6205
Epoch 3, Step 4000, Loss: 3.4564

Epoch